<a href="https://colab.research.google.com/github/mannduuu07-png/korea-fire-frequency-severity-analysis/blob/main/notebooks/04_robustness_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 - Robustness Checks

Tests whether the frequency-severity relationship is stable across
different sampling thresholds and time periods.

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

df = pd.read_parquet('/content/drive/MyDrive/fire_data/processed/cleaned_fire_data.parquet')

def region_stats_excl_worst(data, min_count=100):
    result = []
    for (sido, sigungu), group in data.groupby(['시도', '시군구']):
        total_count = len(group)
        if total_count < min_count:
            continue
        total_casualty = group['인명피해(명)소계'].sum()
        worst_idx = group['인명피해(명)소계'].idxmax()
        worst_val = group.loc[worst_idx, '인명피해(명)소계']
        casualty_excl = total_casualty - worst_val
        result.append({
            '시도': sido, '시군구': sigungu, '화재건수': total_count,
            'casualties_per_100_incl': total_casualty / total_count * 100,
            'casualties_per_100_excl': casualty_excl / (total_count - 1) * 100,
            'worst_incident_casualties': worst_val,
        })
    return pd.DataFrame(result)

def top_n_tuples(region_df, col, n=10):
    top = region_df.nlargest(n, col)
    return set(zip(top['시도'], top['시군구']))

Mounted at /content/drive


## Threshold sensitivity (200 / 300 / 500 cumulative fires)

At every threshold tested, the correlation between fire frequency and
per-fire casualty rate is weak and not statistically significant
(ρ=0.05–0.07, p>0.29 in all cases). This result is consistent
regardless of the sampling threshold chosen.

In [ ]:
for min_c in [200, 300, 500]:
    r = region_stats_excl_worst(df, min_count=min_c)
    c, p = spearmanr(r['화재건수'], r['casualties_per_100_excl'])
    ov = len(top_n_tuples(r, '화재건수') & top_n_tuples(r, 'casualties_per_100_excl'))
    print(f"min_count={min_c}: n_regions={len(r)}, rho={c:.3f}(p={p:.4f}), top10_overlap={ov}")

min_count=200: n_regions=249, rho=0.056(p=0.3772), top10_overlap=0
min_count=300: n_regions=247, rho=0.051(p=0.4237), top10_overlap=0
min_count=500: n_regions=235, rho=0.069(p=0.2937), top10_overlap=0


## Period-split persistence: 2015-2019 vs 2020-2024

2015-2019: ρ=0.043 (p=0.504, not significant)
2020-2024: ρ=0.121 (p=0.058, borderline — not significant at the
conventional α=0.05 threshold, but close enough to warrant caution
in interpretation)

Despite the correlation strengthening somewhat in the more recent
period, only one region — 청주시상당구 is the only region appearing in the high-severity top 10
in both sub-periods — the most consistent candidate across every
sensitivity check performed, though no confidence interval or
null-distribution test has been applied, so chance cannot be formally
ruled out.

In [ ]:
periods = {'2015-2019': range(2015, 2020), '2020-2024': range(2020, 2025)}
top10_by_period = {}
for name, yrs in periods.items():
    sub = df[df['연도'].isin(yrs)]
    r = region_stats_excl_worst(sub, min_count=150)
    c, p = spearmanr(r['화재건수'], r['casualties_per_100_excl'])
    t10 = top_n_tuples(r, 'casualties_per_100_excl')
    top10_by_period[name] = t10
    print(f"{name}: n_regions={len(r)}, rho={c:.3f}(p={p:.4f})")
    print(f"  High-severity top10: {t10}")

persist = top10_by_period['2015-2019'] & top10_by_period['2020-2024']
print(f"\nRegions in high-severity top10 in BOTH periods: {len(persist)} -- {persist}")

2015-2019: n_regions=247, rho=0.043(p=0.5037)
  High-severity top10: {('경기도', '수원시팔달구'), ('대구광역시', '남구'), ('충청북도', '청주시상당구'), ('경기도', '고양시일산동구'), ('경상북도', '포항시북구'), ('강원특별자치도', '삼척시'), ('경상북도', '안동시'), ('대구광역시', '중구'), ('경기도', '안양시동안구'), ('경상북도', '청송군')}
2020-2024: n_regions=247, rho=0.121(p=0.0578)
  High-severity top10: {('충청북도', '청주시상당구'), ('충청북도', '제천시'), ('부산광역시', '사상구'), ('울산광역시', '중구'), ('충청북도', '진천군'), ('강원특별자치도', '영월군'), ('인천광역시', '남동구'), ('경기도', '구리시'), ('인천광역시', '미추홀구'), ('서울특별시', '동대문구')}

Regions in high-severity top10 in BOTH periods: 1 -- {('충청북도', '청주시상당구')}
